In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cell 2
df = pd.read_excel('../data/Customer_Churn_Data_Large.xlsx')

df.head()

,CustomerID,Age,Gender,MaritalStatus,IncomeLevel
0,1,62,M,Single,Low
1,2,65,M,Married,Low
2,3,18,M,Single,Low
3,4,21,M,Widowed,Low
4,5,21,M,Divorced,Medium


In [4]:
df.keys()

Index(['CustomerID', 'Age', 'Gender', 'MaritalStatus', 'IncomeLevel'], dtype='object')

In [5]:
file_path = "../data/Customer_Churn_Data_Large.xlsx"

# Load all sheets
excel_data = pd.read_excel(file_path, sheet_name=None)

# View sheet names
excel_data.keys()

dict_keys(['Customer_Demographics', 'Transaction_History', 'Customer_Service', 'Online_Activity', 'Churn_Status'])

In [6]:
for sheet_name, df in excel_data.items():
    print(f"\n Sheet: {sheet_name}")
    print("Shape:", df.shape)
    display(df.head())


 Sheet: Customer_Demographics
Shape: (1000, 5)


,CustomerID,Age,Gender,MaritalStatus,IncomeLevel
0,1,62,M,Single,Low
1,2,65,M,Married,Low
2,3,18,M,Single,Low
3,4,21,M,Widowed,Low
4,5,21,M,Divorced,Medium



 Sheet: Transaction_History
Shape: (5054, 5)


,CustomerID,TransactionID,TransactionDate,AmountSpent,ProductCategory
0,1,7194,2022-03-27,416.50,Electronics
1,2,7250,2022-08-08,54.96,Clothing
2,2,9660,2022-07-25,197.50,Electronics
3,2,2998,2022-01-25,101.31,Furniture
4,2,1228,2022-07-24,397.37,Clothing



 Sheet: Customer_Service
Shape: (1002, 5)


,CustomerID,InteractionID,InteractionDate,InteractionType,ResolutionStatus
0,1,6363,2022-03-31,Inquiry,Resolved
1,2,3329,2022-03-17,Inquiry,Resolved
2,3,9976,2022-08-24,Inquiry,Resolved
3,4,7354,2022-11-18,Inquiry,Resolved
4,4,5393,2022-07-03,Inquiry,Unresolved



 Sheet: Online_Activity
Shape: (1000, 4)


,CustomerID,LastLoginDate,LoginFrequency,ServiceUsage
0,1,2023-10-21,34,Mobile App
1,2,2023-12-05,5,Website
2,3,2023-11-15,3,Website
3,4,2023-08-25,2,Website
4,5,2023-10-27,41,Website



 Sheet: Churn_Status
Shape: (1000, 2)


,CustomerID,ChurnStatus
0,1,0
1,2,1
2,3,0
3,4,0
4,5,0


In [7]:
for sheet_name, df in excel_data.items():
    print(f"\n {sheet_name} Info")
    print(df.info())
    print(df.describe(include='all'))


 Customer_Demographics Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   CustomerID     1000 non-null   int64 
 1   Age            1000 non-null   int64 
 2   Gender         1000 non-null   object
 3   MaritalStatus  1000 non-null   object
 4   IncomeLevel    1000 non-null   object
dtypes: int64(2), object(3)
memory usage: 39.2+ KB
None
         CustomerID          Age Gender MaritalStatus IncomeLevel
count   1000.000000  1000.000000   1000          1000        1000
unique          NaN          NaN      2             4           3
top             NaN          NaN      F       Widowed        High
freq            NaN          NaN    513           276         349
mean     500.500000    43.267000    NaN           NaN         NaN
std      288.819436    15.242311    NaN           NaN         NaN
min        1.000000    18.000000    NaN        

In [ ]:
# Identify Common Key
for name, df in excel_data.items():
    print(name, df.columns)

Customer_Demographics Index(['CustomerID', 'Age', 'Gender', 'MaritalStatus', 'IncomeLevel'], dtype='object')
Transaction_History Index(['CustomerID', 'TransactionID', 'TransactionDate', 'AmountSpent',
       'ProductCategory'],
      dtype='object')
Customer_Service Index(['CustomerID', 'InteractionID', 'InteractionDate', 'InteractionType',
       'ResolutionStatus'],
      dtype='object')
Online_Activity Index(['CustomerID', 'LastLoginDate', 'LoginFrequency', 'ServiceUsage'], dtype='object')
Churn_Status Index(['CustomerID', 'ChurnStatus'], dtype='object')


In [9]:
# Merge All Sheets



# Load individual sheets
df_demo = excel_data['Customer_Demographics']
df_txn = excel_data['Transaction_History']
df_service = excel_data['Customer_Service']
df_online = excel_data['Online_Activity']
df_churn = excel_data['Churn_Status']

# Aggregate transaction data
txn_agg = df_txn.groupby('CustomerID').agg({
    'AmountSpent': ['sum', 'mean', 'count']
}).reset_index()

txn_agg.columns = [
    'CustomerID',
    'Total_Spend',
    'Avg_Spend',
    'Txn_Count'
]

service_agg = df_service.groupby('CustomerID').agg({
    'InteractionID': 'count'
}).reset_index()

service_agg.columns = ['CustomerID', 'Total_Interactions']

interaction_type = pd.crosstab(
    df_service['CustomerID'],
    df_service['InteractionType']
).reset_index()

df_online['LastLoginDate'] = pd.to_datetime(df_online['LastLoginDate'])

df_online['Days_Since_Last_Login'] = (
    df_online['LastLoginDate'].max() - df_online['LastLoginDate']
).dt.days



# Merge all
df = df_demo.copy()

df = df.merge(txn_agg, on='CustomerID', how='left')
df = df.merge(service_agg, on='CustomerID', how='left')
df = df.merge(interaction_type, on='CustomerID', how='left')
df = df.merge(df_online, on='CustomerID', how='left')
df = df.merge(df_churn, on='CustomerID', how='left')

print("Final Shape:", df.shape)
df.head()

Final Shape: (1000, 17)


,CustomerID,Age,Gender,MaritalStatus,IncomeLevel,Total_Spend,Avg_Spend,Txn_Count,Total_Interactions,Complaint,Feedback,Inquiry,LastLoginDate,LoginFrequency,ServiceUsage,Days_Since_Last_Login,ChurnStatus
0,1,62,M,Single,Low,416.50,416.50000,1,1.0,0.0,0.0,1.0,2023-10-21,34,Mobile App,71,0
1,2,65,M,Married,Low,1547.42,221.06000,7,1.0,0.0,0.0,1.0,2023-12-05,5,Website,26,1
2,3,18,M,Single,Low,1702.98,283.83000,6,1.0,0.0,0.0,1.0,2023-11-15,3,Website,46,0
3,4,21,M,Widowed,Low,917.29,183.45800,5,2.0,0.0,0.0,2.0,2023-08-25,2,Website,128,0
4,5,21,M,Divorced,Medium,2001.49,250.18625,8,NaN,NaN,NaN,NaN,2023-10-27,41,Website,65,0


In [10]:
df['CustomerID'].nunique(), df.shape[0]

(1000, 1000)

In [11]:
# Data Cleaning

# Missing values
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna("Unknown")

# Remove duplicates
df = df.drop_duplicates()

# Optional: sanity check
df.isnull().sum().sort_values(ascending=False).head()

CustomerID       0
Age              0
Gender           0
MaritalStatus    0
IncomeLevel      0
dtype: int64

In [12]:
# Feature Engineering

# Avoid division issues
df['Avg_Spend_Per_Txn'] = df['Total_Spend'] / (df['Txn_Count'] + 1)

# Engagement signal
df['Engagement_Score'] = df['LoginFrequency'] * df['ServiceUsage']

In [13]:
df['Low_Activity_Flag'] = (df['LoginFrequency'] < 5).astype(int)

In [15]:
df.head()

,CustomerID,Age,Gender,MaritalStatus,IncomeLevel,Total_Spend,Avg_Spend,Txn_Count,Total_Interactions,Complaint,Feedback,Inquiry,LastLoginDate,LoginFrequency,ServiceUsage,Days_Since_Last_Login,ChurnStatus,Avg_Spend_Per_Txn,Engagement_Score,Low_Activity_Flag
0,1,62,M,Single,Low,416.50,416.50000,1,1.0,0.0,0.0,1.0,2023-10-21,34,Mobile App,71,0,208.250000,Mobile AppMobile AppMobile AppMobile AppMobile...,0
1,2,65,M,Married,Low,1547.42,221.06000,7,1.0,0.0,0.0,1.0,2023-12-05,5,Website,26,1,193.427500,WebsiteWebsiteWebsiteWebsiteWebsite,0
2,3,18,M,Single,Low,1702.98,283.83000,6,1.0,0.0,0.0,1.0,2023-11-15,3,Website,46,0,243.282857,WebsiteWebsiteWebsite,1
3,4,21,M,Widowed,Low,917.29,183.45800,5,2.0,0.0,0.0,2.0,2023-08-25,2,Website,128,0,152.881667,WebsiteWebsite,1
4,5,21,M,Divorced,Medium,2001.49,250.18625,8,1.5,0.0,0.0,0.0,2023-10-27,41,Website,65,0,222.387778,WebsiteWebsiteWebsiteWebsiteWebsiteWebsiteWebs...,0


In [16]:
df.to_excel('my_dataset.xlsx', index=False)